# Imports

In [ ]:
!pip install torch_geometric

In [2]:
import pandas as pd
import torch
import os
import numpy as np
from torch_geometric.data import Data


In [3]:
# Change directory
os.chdir('/content/drive/MyDrive/nids-mitre/')

!pwd


/content/drive/MyDrive/nids-mitre


In [6]:
# Commented to avoid accidental deletion

# import shutil
# import os

# # Delete old folder
# if os.path.exists("./dataset_processed"):
#     shutil.rmtree("./dataset_processed")
#     print("Folder deleted. Ready to regenerate.")

Folder deleted. Ready to regenerate.


# Configuration

## Set values

In [7]:
# --- CONFIGURATION ---
CSV_PATH = "/content/drive/MyDrive/nids-mitre/data/cicids2018-v3/cicids2018v3_wed2802.csv"
BASE_OUTPUT_DIR = "./dataset_processed"
TIME_WINDOW_SEC = 30  # Each graph represents 30 seconds
CHUNK_SIZE = 1000000   # Rows to read at a time in RAM


## Split train/val/test

In [8]:
# Create directories
splits = ['train', 'val', 'test']
for s in splits:
    os.makedirs(os.path.join(BASE_OUTPUT_DIR, s), exist_ok=True)


In [9]:
cicids2018v3_wed2802_orig = pd.read_csv(CSV_PATH)
cicids2018v3_wed2802 = cicids2018v3_wed2802_orig.copy()
# Convert 'FLOW_START_TIME' to datetime type after loading
cicids2018v3_wed2802['FLOW_START_TIME'] = pd.to_datetime(cicids2018v3_wed2802['FLOW_START_TIME'])

timestamps = cicids2018v3_wed2802['FLOW_START_TIME']
GLOBAL_START_TIME = timestamps.min()
GLOBAL_END_TIME = timestamps.max()
TOTAL_DURATION = GLOBAL_END_TIME - GLOBAL_START_TIME

# Limits for Train/Val/Test
LIMIT_TRAIN = GLOBAL_START_TIME + (TOTAL_DURATION * 0.70)
LIMIT_VAL = GLOBAL_START_TIME + (TOTAL_DURATION * 0.85)

print(f"Global start time: {GLOBAL_START_TIME}")
print(f"Global end time: {GLOBAL_END_TIME}")

print(f"Limit train: {LIMIT_TRAIN}")
print(f"Limit val: {LIMIT_VAL}")


Global start time: 2018-02-28 00:12:55.854000
Global end time: 2018-02-28 23:59:59.321000
Limit train: 2018-02-28 16:51:52.280900
Limit val: 2018-02-28 20:25:55.800950


## Let's check some ports and protocols

In [ ]:
train = cicids2018v3_wed2802[cicids2018v3_wed2802['FLOW_START_TIME'] < LIMIT_TRAIN]

freq_dst_port = train['L4_DST_PORT'].value_counts()

freq_dst_port.head(50)

In [ ]:
train['PROTOCOL'].value_counts()

In [ ]:
train_attack = train[train['Attack'] == 'Infilteration']

freq_dst_port_attack = train_attack['L4_DST_PORT'].value_counts()

freq_dst_port_attack.head(50)

In [ ]:
train_attack['PROTOCOL'].value_counts()

In [ ]:
import gc

del cicids2018v3_wed2802_orig
del cicids2018v3_wed2802
gc.collect()

## Select features

In [10]:
NUMERIC_FEATURES = [
    'IN_BYTES', 'OUT_BYTES', 'IN_PKTS', 'OUT_PKTS',
    'FLOW_DURATION_MILLISECONDS', 'DURATION_IN', 'DURATION_OUT',
    'SRC_TO_DST_IAT_AVG', 'DST_TO_SRC_IAT_AVG',
    'SRC_TO_DST_IAT_STDDEV', 'DST_TO_SRC_IAT_STDDEV',
    'MIN_IP_PKT_LEN', 'MAX_IP_PKT_LEN',
    'RETRANSMITTED_IN_PKTS', 'RETRANSMITTED_OUT_PKTS',
    'TCP_WIN_MAX_IN', 'TCP_WIN_MAX_OUT',
    'TCP_FLAGS', 'MIN_TTL', 'MAX_TTL'
]

CATEGORICAL_FEATURES = [
    'PROTOCOL', 'L4_DST_PORT'
]

# Define functions

## Protocol and Port One-Hot

In [11]:
# One-Hot for Protocol and for Port

def get_protocol_vector(proto):
    """
    One-Hot for Protocol (5 dimensions):
    [TCP(6), UDP(17), ICMP(1 or 58 for ICMPv6), IGMP(2), Other]
    """
    if proto == 6:   return [1, 0, 0, 0, 0] # TCP
    if proto == 17:  return [0, 1, 0, 0, 0] # UDP
    if proto in (1, 58):   return [0, 0, 1, 0, 0] # ICMP or ICMPv6
    if proto == 2:  return [0, 0, 0, 1, 0] # IGMP
    return [0, 0, 0, 0, 1] # Other

def get_port_role_vector(port):
    """
    Classifies the destination port using a 7-dimensional One-Hot vector.
    Optimized for the NF-CSE-CIC-IDS2018-v3 dataset and infiltration attacks.

    [0] Web (HTTP/S, Proxies, RPC-JSON)
    [1] Admin/Remote (SSH, RDP, Telnet, VNC, ADB) - Incluye variantes
    [2] Windows/SMB (NetBIOS, RPC, SMB) - Clave para Mov. Lateral
    [3] DNS/Infra (DNS, LLMNR, DHCP, NTP, SSDP, SIP)
    [4] Database (SQL)
    [5] Other Privileged (< 1024) - Incluye Port 0
    [6] Other High/Ephemeral (>= 1024)
    """

    # 1. Web & HTTP-like
    # 80/443 (Std), 8080/81/3128 (Proxies/Alt), 8545 (Ethereum/RPC)
    if port in [80, 443, 8080, 8443, 81, 3128, 8545]:
        return [1, 0, 0, 0, 0, 0, 0]

    # 2. Admin & Remote Access
    # 22/222/2222 (SSH), 23/2323 (Telnet), 3389/3390/3394 (RDP),
    # 5900/5901 (VNC), 5555 (ADB - Android), 21/2131 (FTP)
    elif port in [22, 222, 2222, 23, 2323, 3389, 3390, 3394, 5900, 5901, 5555, 21, 2131]:
        return [0, 1, 0, 0, 0, 0, 0]

    # 3. Windows Services / SMB
    # 445 (SMB), 135 (RPC Endpoint), 137/138/139 (NetBIOS)
    elif port in [445, 135, 137, 138, 139]:
        return [0, 0, 1, 0, 0, 0, 0]

    # 4. DNS & Infrastructure
    # 53 (DNS), 5355 (LLMNR), 67/547 (DHCP), 123 (NTP), 1900 (SSDP/UPnP), 5060 (SIP)
    elif port in [53, 5355, 67, 547, 123, 1900, 5060]:
        return [0, 0, 0, 1, 0, 0, 0]

    # 5. Database
    # 1433 (MSSQL), 3306 (MySQL)
    elif port in [1433, 3306, 5432, 6379, 27017]:
        return [0, 0, 0, 0, 1, 0, 0]

    # 6. Other Privileged Users (< 1024)
    # Port 0 and other legacy services will be taken offline here
    elif port < 1024:
        return [0, 0, 0, 0, 0, 1, 0]

    # 7. High Ports / Ephemeral (>= 1024)
    else:
        return [0, 0, 0, 0, 0, 0, 1]

In [12]:
edge_attr_dim = len(NUMERIC_FEATURES) + 5 + 7 # one-hot protocol and one-hot port
print(f"edge_attr dimension: {edge_attr_dim}")

edge_attr dimension: 32


## Get an ID for each IP

In [13]:
# Map IP->ID

ip_map = {}
next_id = 0

def get_ip_id(ip_str):
    global next_id
    if ip_str not in ip_map:
        ip_map[ip_str] = next_id
        next_id += 1
    return ip_map[ip_str]


## Create empty graphs

In [14]:
# Auxiliar function for save empty graphs for time windows without nodes

def save_empty_graph(window_id):
    # Determine which split it belongs to based on the window START time
    window_start_time = GLOBAL_START_TIME + pd.Timedelta(seconds=window_id * TIME_WINDOW_SEC)
    if window_start_time < LIMIT_TRAIN: split = 'train'
    elif window_start_time < LIMIT_VAL: split = 'val'
    else: split = 'test'

    # Create empty graph
    # Empty tensors with the correct shape (0 rows, N columns)
    x = torch.empty((0, 16), dtype=torch.float)
    edge_index = torch.empty((2, 0), dtype=torch.long)
    edge_attr = torch.empty((0, edge_attr_dim), dtype=torch.float)
    y = torch.empty((0), dtype=torch.float) # No labels

    data = Data(x=x, edge_index=edge_index, edge_attr=edge_attr, y=y)
    data.global_node_ids = torch.empty((0), dtype=torch.long)
    data.timestamp = window_start_time
    data.is_empty = True # Flag for model

    filename = f"graph_{int(window_id):06d}.pt"
    torch.save(data, os.path.join(BASE_OUTPUT_DIR, split, filename))


## Create graphs (not empty)

In [15]:
def save_window_group(window_id, df_group):
    # Determine which split it belongs to based on the window START time
    window_start_time = GLOBAL_START_TIME + pd.Timedelta(seconds=window_id * TIME_WINDOW_SEC)
    if window_start_time < LIMIT_TRAIN: split = 'train'
    elif window_start_time < LIMIT_VAL: split = 'val'
    else: split = 'test'

    # Graph nodes: GLOBAL IDS
    src_global = [get_ip_id(ip) for ip in df_group['IPV4_SRC_ADDR']]
    dst_global = [get_ip_id(ip) for ip in df_group['IPV4_DST_ADDR']]

    # MAP: GLOBAL -> LOCAL
    ## Unique nodes that are in THIS window
    unique_global_nodes = sorted(list(set(src_global + dst_global)))

    ## Dictionary: {Global_ID: Local_ID (0, 1, 2...)}
    global_to_local = { gid: lid for lid, gid in enumerate(unique_global_nodes) }

    # RE-INDEX EDGES (Using Local IDs)
    src_local = [global_to_local[gid] for gid in src_global]
    dst_local = [global_to_local[gid] for gid in dst_global]

    edge_index = torch.tensor([src_local, dst_local], dtype=torch.long)

    # Features and normalization
    ## 1. Numeric Features (Log1p)
    numeric_tensor = torch.tensor(df_group[NUMERIC_FEATURES].values.astype('float32'))
    numeric_tensor = torch.log1p(numeric_tensor)

    ## 2. Dst Port (One-Hot)
    port_list = df_group['L4_DST_PORT'].apply(get_port_role_vector).tolist()
    port_tensor = torch.tensor(port_list, dtype=torch.float32)

    ## 3. Protocol (One-Hot)
    proto_list = df_group['PROTOCOL'].apply(get_protocol_vector).tolist()
    proto_tensor = torch.tensor(proto_list, dtype=torch.float32)

    ## 4. Concatenate everything to form the final edge_attr
    edge_attr = torch.cat([port_tensor, proto_tensor, numeric_tensor], dim=1)

    # Nodes (Identity Agnostic)
    num_nodes = len(unique_global_nodes)
    x = torch.ones((num_nodes, 16), dtype=torch.float)

    # Target
    y = torch.tensor(
        df_group['Attack'].apply(lambda x: 1 if x == 'Infilteration' else 0).values,
        dtype=torch.float
    )

    # Put together
    data = Data(x=x, edge_index=edge_index, edge_attr=edge_attr, y=y)

    # IMPORTANT: We keep a record of the actual global IDs
    # This is what the ST-GNN will use to retrieve the correct memory
    data.global_node_ids = torch.tensor(unique_global_nodes, dtype=torch.long)
    data.timestamp = window_start_time

    # Save: We use the window_id as the name to maintain strict order
    # graph_00000.pt, graph_00001.pt ...
    filename = f"graph_{int(window_id):06d}.pt"
    torch.save(data, os.path.join(BASE_OUTPUT_DIR, split, filename))

# Complete process loop

In [16]:
# PROCESS LOOP
expected_window_id = 0 # Counter for tracking continuity
buffer_df = pd.DataFrame()
print("Processing chunks...")

# Iterador de chunks
chunk_iterator = pd.read_csv(CSV_PATH, chunksize=CHUNK_SIZE)

for chunk_idx, chunk in enumerate(chunk_iterator):
    # 1. Join with previous buffer (if one exists)
    df = pd.concat([buffer_df, chunk])
    df = df[ (df['IPV4_SRC_ADDR'] != '0.0.0.0') & (df['IPV4_DST_ADDR'] != '0.0.0.0') ] # Eliminate 0.0.0.0 (self-loops)
    df['FLOW_START_TIME'] = pd.to_datetime(df['FLOW_START_TIME'])

    # 2. Assign Window ID to each row (Vectorized = Very Fast)
    # Subtract the global starting value and divide by 30. The floor (int) gives us the ID.
    df['window_id'] = ((df['FLOW_START_TIME'] - GLOBAL_START_TIME) // pd.Timedelta(seconds=TIME_WINDOW_SEC)).astype(int)

    # 3. Identify which windows we have in this chunk
    present_windows = df['window_id'].unique()
    present_windows.sort()

    if len(present_windows) > 0:
        # 4. SECURITY STRATEGY:
        # The last window in this chunk may be incomplete.
        # (Your data may continue in the next chunk.)
        # Therefore, we process all windows EXCEPT the last one.
        last_window_id = present_windows[-1]

        # Separate: Complete vs Incomplete
        df_complete = df[df['window_id'] < last_window_id]
        df_incomplete = df[df['window_id'] == last_window_id]

        # 5. Process entire groups
        for win_id, group in df_complete.groupby('window_id'):
            # 1. Fill in the gaps if we skip numbers
            while expected_window_id < win_id:
                save_empty_graph(expected_window_id)
                expected_window_id += 1

            # 2. Save the current window (which does contain data)
            save_window_group(win_id, group)
            expected_window_id += 1

        # 6. Save the incomplete part in the buffer for the next chunk
        buffer_df = df_incomplete
    else:
        # Rare case: The entire chunk belongs to a window that was already in the buffer
        buffer_df = df

    print(f"Chunk {chunk_idx} processed. Buffer size: {len(buffer_df)}")


# FINAL CLEANING
# At the end, what remains in the buffer is the last valid window
if not buffer_df.empty:
    print("Processing last buffer...")
    for win_id, group in buffer_df.groupby('window_id'):
        # Fill in any gaps before the last window, if there are any
        while expected_window_id < win_id:
            save_empty_graph(expected_window_id)
            expected_window_id += 1
        save_window_group(win_id, group)
        expected_window_id += 1

# Fill any gaps after the last processed graph until the end of the dataset
# Let's calculate the ID of the last possible window.
final_possible_window_id = ((GLOBAL_END_TIME - GLOBAL_START_TIME) // pd.Timedelta(seconds=TIME_WINDOW_SEC))
while expected_window_id <= final_possible_window_id:
    save_empty_graph(expected_window_id)
    expected_window_id += 1

print("Dataset generation complete!")

Processing chunks...
Chunk 0 processed. Buffer size: 984
Chunk 1 processed. Buffer size: 308
Chunk 2 processed. Buffer size: 3
Processing last buffer...
Dataset generation complete!


# Some verifications

In [17]:
import torch
import os
import random

ROOT_DIR = "./dataset_processed/train"

# Choose a random file that is NOT empty (to see real data)
files = sorted([f for f in os.listdir(ROOT_DIR) if f.endswith('.pt')])
sample_data = None
for f in files:
    data = torch.load(os.path.join(ROOT_DIR, f), weights_only=False)
    if data.x.shape[0] > 0: # If it has nodes
        sample_data = data
        print(f"Analyzing file: {f}")
        break

if sample_data:
    print("="*40)
    print(f"Available keys: {sample_data.keys}")
    print(f"Nodes (x): {sample_data.x.shape} -> (Num Nodes, 16 features dummy)")
    print(f"Edges (edge_index): {sample_data.edge_index.shape} -> (2, Num Flows)")
    print(f"Edge features (edge_attr): {sample_data.edge_attr.shape} -> (Num Flows, {sample_data.edge_attr.shape[1]})")
    print(f"Labels (y): {sample_data.y.shape} -> (Num Flows,)")
    print(f"Timestamp: {sample_data.timestamp}")
    print(f"Is it empty?: {getattr(sample_data, 'is_empty', False)}")

    # Check
    num_edges = sample_data.edge_index.shape[1]
    num_labels = sample_data.y.shape[0]
    num_attrs = sample_data.edge_attr.shape[0]

    if num_edges == num_labels == num_attrs:
        print("✅ Number of edges matches features and labels.")
    else:
        print("❌ ERROR: Mismatch in edge dimensions.")

    # Check if there are any attacks on this snapshot
    attacks = sample_data.y.sum().item()
    print(f"Number of malicious flows in this graph: {int(attacks)}")
else:
    print("Only empty graphs or no files.")

Analyzing file: graph_000000.pt
Available keys: <bound method BaseData.keys of Data(x=[9, 16], edge_index=[2, 8], edge_attr=[8, 32], y=[8], global_node_ids=[9], timestamp=2018-02-28 00:12:55.854000)>
Nodes (x): torch.Size([9, 16]) -> (Num Nodes, 16 features dummy)
Edges (edge_index): torch.Size([2, 8]) -> (2, Num Flows)
Edge features (edge_attr): torch.Size([8, 32]) -> (Num Flows, 32)
Labels (y): torch.Size([8]) -> (Num Flows,)
Timestamp: 2018-02-28 00:12:55.854000
Is it empty?: False
✅ Number of edges matches features and labels.
Number of malicious flows in this graph: 0


In [18]:
import pandas as pd

def check_temporal_continuity(split_name='train'):
    dir_path = os.path.join("./dataset_processed", split_name)
    files = sorted([f for f in os.listdir(dir_path) if f.endswith('.pt')])

    print(f"\nChecking continuity in split: {split_name} ({len(files)} files)...")

    prev_id = -1
    prev_time = None

    errors = 0
    empty_graphs = 0

    for f in files:
        # Extract ID from name "graph_000123.pt"
        curr_id = int(f.split('_')[1].split('.')[0])

        data = torch.load(os.path.join(dir_path, f), weights_only=False)
        curr_time = data.timestamp

        # 1. Verify consecutive ID
        if prev_id != -1:
            if curr_id != prev_id + 1:
                print(f"❌ JUMP DETECTED: We went from {prev_id} to {curr_id}. Files are missing.")
                errors += 1

        # 2. Verify Time (must be exactly +30s)
        # Note: curr_time can be a timestamp or string; we guarantee conversion
        if prev_time is not None:
            diff = (curr_time - prev_time).total_seconds()
            if diff != 30.0:
                 print(f"⚠️ TIME ALERT: Graph {curr_id} jump of {diff}s (expected 30s)")

        # 3. Count gaps
        if data.x.shape[0] == 0:
            empty_graphs += 1

        prev_id = curr_id
        prev_time = curr_time

    if errors == 0:
        print(f"✅ CORRECT numerical sequence (consecutive IDs).")
    print(f"ℹ️ Total empty graphs found: {empty_graphs}")

#check_temporal_continuity('train')
check_temporal_continuity('test')


Checking continuity in split: test (429 files)...
✅ CORRECT numerical sequence (consecutive IDs).
ℹ️ Total empty graphs found: 0


In [19]:
def check_content_stats(split_name='train'):
    dir_path = os.path.join("./dataset_processed", split_name)
    files = sorted([f for f in os.listdir(dir_path) if f.endswith('.pt')])

    total_flows = 0
    total_attacks = 0
    has_nans = False

    # Random sampling of 50 graphs to avoid saturation
    sample_files = random.sample(files, min(len(files), 50))

    print(f"\nAnalyzing content (Sample of {len(sample_files)} graphs)...")

    for f in sample_files:
        data = torch.load(os.path.join(dir_path, f), weights_only=False)

        if data.x.shape[0] == 0: continue # Jumping gaps

        # Check NaNs
        if torch.isnan(data.edge_attr).any() or torch.isnan(data.x).any():
            print(f"❌ NaN detected in {f}")
            has_nans = True

        # Check Infs
        if torch.isinf(data.edge_attr).any():
             print(f"❌ Inf detected in {f} (Check log1p)")

        total_flows += data.y.shape[0]
        total_attacks += data.y.sum().item()

    print(f"Total Flows Analyzed: {total_flows}")
    print(f"Total Attacks Found: {int(total_attacks)}")
    print(f"Percentage of Attacks: {(total_attacks/total_flows)*100:.4f}%")

    if not has_nans:
        print("✅ No NaNs were detected in the sample.")

check_content_stats('train')


Analyzing content (Sample of 50 graphs)...
Total Flows Analyzed: 32832
Total Attacks Found: 1002
Percentage of Attacks: 3.0519%
✅ No NaNs were detected in the sample.


In [20]:
import torch
import os
import random

def check_multigraph_stats(split='train', num_samples=100):
    dir_path = os.path.join("./dataset_processed", split)
    files = sorted([f for f in os.listdir(dir_path) if f.endswith('.pt')])

    # We took a random sample
    sample_files = random.sample(files, min(len(files), num_samples))

    multigraph_count = 0
    max_parallel_edges = 0

    print(f"Analyzing {len(sample_files)} graphs in search of parallel edges...")

    for f in sample_files:
        data = torch.load(os.path.join(dir_path, f), weights_only=False)

        if data.x.shape[0] == 0: continue

        # edge_index has the form [2, E]. We transpose it to [E, 2] to see pairs.
        edges_T = data.edge_index.t()

        # Find unique pairs and count their occurrences
        # return_counts=True tells how many times each pair appears
        unique_edges, counts = torch.unique(edges_T, dim=0, return_counts=True)

        # If the number of unique pairs is LESS than the total number of edges,
        # it means there are repetitions (multigraph).
        num_unique = unique_edges.shape[0]
        num_total = edges_T.shape[0]

        if num_total > num_unique:
            multigraph_count += 1
            current_max = counts.max().item()
            if current_max > max_parallel_edges:
                max_parallel_edges = current_max

            # (Optional) Print an example the first time it is found
            if multigraph_count == 1:
                print(f"🔎 Found in {f}:")
                print(f"   - Total Flows: {num_total}")
                print(f"   - Unique Connections (IP-IP): {num_unique}")
                print(f"   - Difference (Parallel edges): {num_total - num_unique}")
                print(f"   - Maximum parallel flows between a pair of IPs: {current_max}")

    print("-" * 30)
    print(f"Graphs with multiple edges: {multigraph_count} of {len(sample_files)}")
    print(f"Highest number of parallel flows seen between two IPs: {max_parallel_edges}")

check_multigraph_stats('train')

Analyzing 100 graphs in search of parallel edges...
🔎 Found in graph_000403.pt:
   - Total Flows: 4
   - Unique Connections (IP-IP): 2
   - Difference (Parallel edges): 2
   - Maximum parallel flows between a pair of IPs: 3
------------------------------
Graphs with multiple edges: 31 of 100
Highest number of parallel flows seen between two IPs: 346
